# Accessing BigQuery Datasets in Workbench

**Dataset:** `wb-crisp-bean-1269.temporary_data`

This notebook demonstrates how to query a BigQuery dataset from a Verily Workbench
cloud environment using the `google-cloud-bigquery` Python client library.

The dataset contains national wastewater pathogen surveillance data, including
measurements from over 2,000 treatment plants across the United States.

## Overview

BigQuery datasets attached to your Workbench workspace are accessible from notebook
cloud environments using standard Google Cloud client libraries. Authentication is
handled automatically by the environment — no API keys or service account setup is
required.

### Objective

Use this notebook to learn how to:

- Connect to a BigQuery dataset from Python
- List available tables in a dataset
- Inspect table schema and metadata
- Run SQL queries and load results into a pandas DataFrame
- Compute summary statistics with aggregation queries

### Costs

This notebook runs a small number of queries against a ~400 MB dataset. The total
data scanned will be well under 1 GB.

## Setup

Import the BigQuery client library and configure the dataset reference.

In [ ]:
import os

from google.cloud import bigquery

DATA_PROJECT = "wb-crisp-bean-1269"
BILLING_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", DATA_PROJECT)
DATASET = "temporary_data"
DATASET_REF = f"{DATA_PROJECT}.{DATASET}"

client = bigquery.Client(project=BILLING_PROJECT)

## 1. List Tables in the Dataset

A BigQuery dataset can contain multiple tables. The cell below lists all tables
available in the dataset.

In [2]:
tables = list(client.list_tables(DATASET_REF))

print(f"Found {len(tables)} table(s) in {DATASET_REF}:\n")
for t in tables:
    print(f"  - {t.table_id} ({t.table_type})")

Found 1 table(s) in wb-crisp-bean-1269.temporary_data:

  - all (TABLE)


## 2. Inspect Table Schema

Before querying, it is useful to understand the structure of the data. The cell
below retrieves metadata for the table, including the number of rows, the size
on disk, and the full column schema.

In [3]:
table = client.get_table(f"{DATASET_REF}.all")

print(f"Table:  {table.full_table_id}")
print(f"Rows:   {table.num_rows:,}")
print(f"Size:   {table.num_bytes / 1e6:.1f} MB")
print(f"\nSchema ({len(table.schema)} columns):")
for field in table.schema:
    print(f"  {field.name:30s} {field.field_type:10s}  {field.description or ''}")

Table:  wb-crisp-bean-1269:temporary_data.all
Rows:   2,549,153
Size:   393.1 MB

Schema (10 columns):
  source                         STRING      Data source identifier, i.e. 'Verily', 'Email', 'cdc.data.gov'
  link                           STRING      URL to data source
  plant_name                     STRING      Full name of the wastewater treatment plant
  plant_region                   STRING      Region (e.g. state or territory) where the plant is located
  sewershed_population           INTEGER     Population served by this plant's sewershed
  sample_collection_date         DATE        Date when the wastewater sample was collected
  pathogen                       STRING      CDC-style lowercase pathogen name (e.g. sars-cov-2, fluav, rsv)
  unit                           STRING      Unit of measurement for the sample
  copies_per_unit                FLOAT       How many copies of pathogen's amplicon were present per unit in the sample
  copies_per_pmmov               FLOAT    

## 3. Preview Data

Run a simple `SELECT *` query with a `LIMIT` clause to preview the first few rows.
The results are returned as a pandas DataFrame for convenient display.

In [4]:
query = f"""
SELECT *
FROM `{DATASET_REF}.all`
LIMIT 10
"""

df = client.query(query).to_dataframe()
df

,source,link,plant_name,plant_region,sewershed_population,sample_collection_date,pathogen,unit,copies_per_unit,copies_per_pmmov
0,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1000,Minnesota,23378,2026-03-23,SARS-CoV-2,liter,0.00000,0.00044
1,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1000,Minnesota,23378,2026-03-23,Influenza A,liter,0.00000,0.00044
2,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1000,Minnesota,23378,2026-03-23,RSV,liter,0.00000,0.00044
3,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1002,Minnesota,18544,2026-03-23,Influenza A,liter,0.00000,0.00027
4,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1002,Minnesota,18544,2026-03-23,SARS-CoV-2,liter,8136.00000,0.00027
5,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1002,Minnesota,18544,2026-03-23,RSV,liter,0.00000,0.00027
6,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1003,Minnesota,16000,2026-03-23,SARS-CoV-2,gram,186115.99058,0.00049
7,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1003,Minnesota,16000,2026-03-23,RSV,gram,39153.88005,0.00010
8,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1011,Minnesota,13628,2026-03-23,Influenza A,liter,0.00000,0.00012
9,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillanc...,1011,Minnesota,13628,2026-03-23,SARS-CoV-2,liter,8435.00000,0.00012


## 4. Summary Statistics

Aggregation queries are a good way to get a high-level picture of the dataset.
The query below computes counts, distinct values, and the date range of sample
collections.

In [5]:
summary_query = f"""
SELECT
  COUNT(*)                       AS total_rows,
  COUNT(DISTINCT plant_name)     AS distinct_plants,
  COUNT(DISTINCT pathogen)       AS distinct_pathogens,
  COUNT(DISTINCT plant_region)   AS distinct_regions,
  MIN(sample_collection_date)    AS earliest_date,
  MAX(sample_collection_date)    AS latest_date
FROM `{DATASET_REF}.all`
"""

summary = client.query(summary_query).to_dataframe()
summary.T

,0
total_rows,2549153
distinct_plants,2110
distinct_pathogens,27
distinct_regions,52
earliest_date,2020-01-14
latest_date,2026-06-01


## Next Steps

You now have the building blocks to query any BigQuery dataset in your Workbench
workspace. From here you can:

- Write more complex SQL queries with `WHERE`, `GROUP BY`, and `JOIN` clauses
- Use `pandas-gbq` or `%%bigquery` cell magics as alternative query interfaces
- Save query results to new BigQuery tables or export to Cloud Storage

In [ ]:
print("Done. All queries completed successfully.")